# Trabajo Guia de Laboratorio 01

In [5]:
#importamos librerias

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:
#cargamos el data frame y mostramos la cantidad de datos
db_path = os.path.join('BD', 'DETALLE_INVERSIONES.csv')
df = pd.read_csv(db_path, low_memory = False)

filas, columnas = df.shape
print("Filas:", filas)
print("Columnas:", columnas)

Filas: 260009
Columnas: 68


## 4 . PERFILADO

### NIVEL 1 — Identidad: Validación del Código Único (CUI)
Antes de procesar cualquier dato financiero o técnico, debemos asegurar que el registro represente un proyecto real y rastreable.

**Lógica del Perfilado:**
* `mask_cui_invalido`: Identifica registros donde el `CODIGO_UNICO` es nulo (vacío) o no cumple con el estándar técnico de 7 dígitos (rango entre 1,000,000 y 9,999,999).
* `error_CU`: Almacena los registros descartados. Si el ID está mal, el registro no tiene trazabilidad.
* `df_valido`: Es nuestro "dataset maestro" filtrado que pasará al siguiente nivel.

**Variables involucradas:**
* `CODIGO_UNICO`: Identificador numérico clave.

In [7]:
# Convertimos a numérico para validar rangos
df['CODIGO_UNICO'] = pd.to_numeric(df['CODIGO_UNICO'], errors='coerce')

# Definimos las condiciones de error
mask_cui_invalido = df['CODIGO_UNICO'].isna() | ~df['CODIGO_UNICO'].between(1000000, 9999999)

# Separamos la data
error_CU = df[mask_cui_invalido].copy()
df_valido = df[~mask_cui_invalido].copy()

print(f"Nivel 1 finalizado:")
print(f"- Registros inválidos (error_CU): {len(error_CU)}")
print(f"- Registros con identidad confirmada (df_valido): {len(df_valido)}")

Nivel 1 finalizado:
- Registros inválidos (error_CU): 67
- Registros con identidad confirmada (df_valido): 259942


### NIVEL 2 — Lógica de Negocio: Estado vs. Situación
En este nivel verificamos la coherencia administrativa. Un proyecto no puede estar "operando" si legalmente no ha sido aprobado.

**Lógica del Perfilado:**
* `mask_estado_incoherente`: Detecta proyectos que figuran como **ACTIVO** en su estado, pero cuya situación técnica no es **VIABLE**. En la gestión pública, la viabilidad es el "permiso legal" indispensable para ejecutar recursos.
* `error_estado`: Captura estas inconsistencias donde el sistema muestra actividad en proyectos no autorizados.

**Variables involucradas:**
* `ESTADO`: Situación administrativa (Activo, Desactivado, etc.).
* `SITUACION`: Condición técnica (Viable, Aprobado, En Formulación).

In [8]:
# Normalizamos textos para evitar errores por espacios o minúsculas
df_valido['ESTADO'] = df_valido['ESTADO'].astype(str).str.strip().str.upper()
df_valido['SITUACION'] = df_valido['SITUACION'].astype(str).str.strip().str.upper()

# Filtro de inconsistencia: Activo pero no Viable
mask_estado_incoherente = (df_valido['ESTADO'] == 'ACTIVO') & (df_valido['SITUACION'] != 'VIABLE')

error_estado = df_valido[mask_estado_incoherente].copy()
df_proyectos_coherentes = df_valido[~mask_estado_incoherente].copy()

print(f"Nivel 2 finalizado:")
print(f"- Proyectos con estado incoherente: {len(error_estado)}")
print(f"- Proyectos con lógica de negocio aprobada: {len(df_proyectos_coherentes)}")

Nivel 2 finalizado:
- Proyectos con estado incoherente: 59843
- Proyectos con lógica de negocio aprobada: 200099


#### NIVEL 3 — Línea de Tiempo: Cronología del Proyecto
Validamos que el ciclo de vida del proyecto siga un orden cronológico lógico. El tiempo en la inversión pública es lineal y sucesivo.

**Lógica del Perfilado:**
* `error_fechas`: Registros donde la `FECHA_REGISTRO` es posterior a la `FECHA_VIABILIDAD`. Es imposible aprobar un proyecto antes de que sea registrado como idea.
* También se evalúa que la ejecución (`FEC_INI_EJECUCION`) no haya iniciado antes de obtener la viabilidad.

**Variables involucradas:**
* `FECHA_REGISTRO`: Nacimiento de la idea.
* `FECHA_VIABILIDAD`: Aprobación técnica/económica.
* `FEC_INI_EJECUCION`: Inicio de obra física.

In [9]:
# Conversión masiva a formato fecha
for col in ['FECHA_REGISTRO', 'FECHA_VIABILIDAD', 'FEC_INI_EJECUCION']:
    df_proyectos_coherentes[col] = pd.to_datetime(df_proyectos_coherentes[col], errors='coerce')

# Validación de orden lógico
mask_cronologia_erronea = (df_proyectos_coherentes['FECHA_REGISTRO'] > df_proyectos_coherentes['FECHA_VIABILIDAD']) | \
                          (df_proyectos_coherentes['FEC_INI_EJECUCION'] < df_proyectos_coherentes['FECHA_VIABILIDAD'])

error_cronologia = df_proyectos_coherentes[mask_cronologia_erronea].copy()

print(f"Nivel 3 finalizado:")
print(f"- Proyectos con fechas imposibles: {len(error_cronologia)}")

Nivel 3 finalizado:
- Proyectos con fechas imposibles: 8091


### NIVEL 4 — Consistencia Financiera: Análisis de Saldos
Analizamos si los montos de dinero pendientes de ejecutar tienen coherencia con el presupuesto total asignado al proyecto.

**Lógica del Perfilado:**
* `error_financiero_critico`: Ocurre cuando el `SALDO_EJECUTAR` es mayor al `COSTO_ACTUALIZADO`. No puedes tener pendiente más dinero del que cuesta el proyecto total.
* `advertencia_sobrecosto`: Cuando el saldo pendiente ya superó el `MONTO_VIABLE` original, indicando que el proyecto se ha encarecido más de lo previsto inicialmente.

**Variables involucradas:**
* `SALDO_EJECUTAR`: Dinero que falta gastar.
* `COSTO_ACTUALIZADO`: Valor total del proyecto hoy.
* `MONTO_VIABLE`: Valor original aprobado.

In [10]:
# Aseguramos formato numérico en montos
for col in ['SALDO_EJECUTAR', 'COSTO_ACTUALIZADO', 'MONTO_VIABLE']:
    df_proyectos_coherentes[col] = pd.to_numeric(df_proyectos_coherentes[col], errors='coerce')

# Detección de errores críticos (Saldo > Total)
mask_financiero_critico = df_proyectos_coherentes['SALDO_EJECUTAR'] > df_proyectos_coherentes['COSTO_ACTUALIZADO']
error_financiero = df_proyectos_coherentes[mask_financiero_critico].copy()

# Detección de advertencias (Saldo > Viable inicial)
mask_sobrecosto = df_proyectos_coherentes['SALDO_EJECUTAR'] > df_proyectos_coherentes['MONTO_VIABLE']
advertencia_financiera = df_proyectos_coherentes[mask_sobrecosto].copy()

print(f"Nivel 4 finalizado:")
print(f"- Errores financieros críticos: {len(error_financiero)}")
print(f"- Alertas por sobrecostos detectados: {len(advertencia_financiera)}")

Nivel 4 finalizado:
- Errores financieros críticos: 1
- Alertas por sobrecostos detectados: 26323


### NIVEL 5 — Realidad Física: Porcentajes de Avance
Finalmente, verificamos los indicadores de progreso. Los porcentajes son valores matemáticos acotados.

**Lógica del Perfilado:**
* `error_avance`: Identifica valores fuera del rango [0 - 100]. 
* **Nota técnica:** Valores extremadamente altos (como 200,000,000) sugieren que el usuario ingresó el monto ejecutado en soles en la celda destinada al porcentaje (error de campo).

**Variables involucradas:**
* `AVANCE_FISICO`: Progreso de la obra en el campo.
* `AVANCE_EJECUCION`: Progreso del gasto financiero.

In [11]:
# Validación de rango porcentual [0-100]
df_proyectos_coherentes['AVANCE_FISICO'] = pd.to_numeric(df_proyectos_coherentes['AVANCE_FISICO'], errors='coerce')

mask_avance_imposible = (df_proyectos_coherentes['AVANCE_FISICO'] < 0) | (df_proyectos_coherentes['AVANCE_FISICO'] > 100)
error_avance = df_proyectos_coherentes[mask_avance_imposible].copy()

print(f"Nivel 5 finalizado:")
print(f"- Registros con porcentajes imposibles: {len(error_avance)}")
if len(error_avance) > 0:
    print(f"- Valor máximo detectado erróneamente: {error_avance['AVANCE_FISICO'].max()}")

Nivel 5 finalizado:
- Registros con porcentajes imposibles: 140
- Valor máximo detectado erróneamente: 61325049.14


### NIVEL 6 — Resumen de Hallazgos: Impacto en la Calidad de Datos
En este nivel final, consolidamos todas las métricas de error detectadas para obtener una visión general de la salud del dataset. Este paso es crucial para determinar si la información es apta para la toma de decisiones estratégicas en el proyecto.

**Lógica del Perfilado:**
* **Consolidación:** Agrupamos los conteos de registros afectados en cada nivel previo (Identidad, Lógica, Cronología, Financiero y Física).
* **Cálculo de Representatividad:** Determinamos el `% del total` para cada incidencia. Esto permite priorizar qué errores requieren una limpieza inmediata.
* `resumen_perfilado`: Genera una tabla técnica con las categorías y descripciones de los fallos encontrados.


**Variables involucradas:**
* `resumen_perfilado`: DataFrame que centraliza los resultados de los 5 niveles técnicos previos.
* `% del total`: Cálculo de representatividad que mide el impacto de cada inconsistencia sobre el universo total de datos.
* `total`: Variable de control que almacena el número total de registros procesados.

In [12]:

total = len(df)

# Creamos la estructura del reporte final basada en los niveles previos
resumen_perfilado = pd.DataFrame([
    (1, 'Identidad',          'CUI nulo o sin 7 dígitos',              len(error_CU)),
    (2, 'Lógica de Negocio',   'Activo sin Situación Viable',           len(error_estado)),
    (3, 'Cronología',          'Fechas fuera de orden lógico',          len(error_cronologia)),
    (4, 'Financiero - Crítico','Saldo > Costo actualizado',             len(error_financiero)),
    (4, 'Financiero - Alerta', 'Saldo > Monto viable',                  len(advertencia_financiera)),
    (5, 'Realidad Física',     'Avance fuera de rango (0-100)',         len(error_avance)),
], columns=['Nivel', 'Categoría', 'Descripción', 'Registros_afectados'])

# Cálculo del impacto porcentual
resumen_perfilado['% del total'] = (
    resumen_perfilado['Registros_afectados'] / total * 100
).round(2)

# Impresión del reporte técnico en formato plano
print("=====================================================================")
print("           REPORTE FINAL DE PERFILADO DE DATOS                       ")
print("=====================================================================")
print(resumen_perfilado.to_string(index=False))
print("=====================================================================")
print(f"Total de registros procesados: {total}")

           REPORTE FINAL DE PERFILADO DE DATOS                       
 Nivel            Categoría                   Descripción  Registros_afectados  % del total
     1            Identidad      CUI nulo o sin 7 dígitos                   67         0.03
     2    Lógica de Negocio   Activo sin Situación Viable                59843        23.02
     3           Cronología  Fechas fuera de orden lógico                 8091         3.11
     4 Financiero - Crítico     Saldo > Costo actualizado                    1         0.00
     4  Financiero - Alerta          Saldo > Monto viable                26323        10.12
     5      Realidad Física Avance fuera de rango (0-100)                  140         0.05
Total de registros procesados: 260009


## 5 . LIMPIEZA DE DATOS


### Descripción: 
en este punto, limpiaremos los **3 niveles** cuyos valores hayan guardado la mayor cantidad de errores en relacion al % de la base de datos, posterior a eso lo guardaremos en una copia limpia de la base de datos.

In [13]:
#hacemos la copia de la db
clean_df = df.copy()

## NIVEL 2 — Consistencia de Estado y Situación

En este nivel se evalúa la coherencia lógica entre las variables categóricas que describen el estado del proyecto.

### Lógica de la limpieza

* Se normalizan las variables `ESTADO` y `SITUACION` para evitar inconsistencias de formato.
* Se define la regla de negocio: un proyecto con estado "ACTIVO" debe tener situación "VIABLE".
* Los registros que no cumplen esta condición se consideran inconsistentes y son eliminados.

### Variables involucradas

* `ESTADO`: Estado actual del proyecto.
* `SITUACION`: Condición de viabilidad del proyecto.

### Motivos de limpieza:
Los registros con estado ACTIVO y situación distinta de VIABLE representan una inconsistencia lógica, ya que un proyecto no debería ejecutarse sin haber sido previamente aprobado.





In [15]:
# nivel 2 - consistencia de estado 

# normalizamos texto
clean_df['ESTADO'] = clean_df['ESTADO'].astype(str).str.strip().str.upper()
clean_df['SITUACION'] = clean_df['SITUACION'].astype(str).str.strip().str.upper()

# condiciones
mask_estado_incoherente = (
    (clean_df['ESTADO'] == 'ACTIVO') &
    (clean_df['SITUACION'] != 'VIABLE')
)

# 3. Separar errores
error_estado_df = clean_df[mask_estado_incoherente].copy()

# 4. Excluirlos del dataset limpio
clean_df = clean_df[~mask_estado_incoherente].copy()

# 5. Resultados
print("Limpieza: Consistencia ESTADO - SITUACION")
print(f"Registros eliminados: {len(error_estado_df)}")
print(f"Registros finales: {len(clean_df)}")

Limpieza: Consistencia ESTADO - SITUACION
Registros eliminados: 59843
Registros finales: 200166




## NIVEL 4 — Consistencia Financiera

En este nivel se valida la coherencia de las variables financieras del proyecto, asegurando que los montos sean consistentes entre sí.

### Lógica de la limpieza

* Se convierten las variables `SALDO_EJECUTAR`, `COSTO_ACTUALIZADO` y `MONTO_VIABLE` a formato numérico.
* Se define una regla crítica: el saldo por ejecutar no puede ser mayor al costo actualizado.
* Se define una regla adicional: el saldo no debe superar el monto viable del proyecto.
* Los registros que incumplen cualquiera de estas condiciones se consideran inválidos y son eliminados.

### Variables involucradas

* `SALDO_EJECUTAR`: Monto pendiente de ejecución.
* `COSTO_ACTUALIZADO`: Costo total actualizado del proyecto.
* `MONTO_VIABLE`: Monto aprobado inicialmente.

### Motivos de limpieza:
Los registros donde el saldo supera el costo total o el monto viable reflejan inconsistencias financieras, ya que implican montos imposibles dentro de la lógica presupuestaria del proyecto.


In [21]:
# nivel 3 - consistencia cronologica

# conversion a formato fecha
for col in ['FECHA_REGISTRO', 'FECHA_VIABILIDAD', 'FEC_INI_EJECUCION']:
    clean_df[col] = pd.to_datetime(clean_df[col], errors='coerce')

# condiciones
mask_cronologia_erronea = (
    (clean_df['FECHA_REGISTRO'] > clean_df['FECHA_VIABILIDAD']) |
    (clean_df['FEC_INI_EJECUCION'] < clean_df['FECHA_VIABILIDAD'])
)

# Separar errores
error_cronologia_df = clean_df[mask_cronologia_erronea].copy()

# Eliminar registros invalidos
clean_df = clean_df[~mask_cronologia_erronea].copy()

# Resultados
print("Limpieza: Consistencia cronológica")
print(f"Registros eliminados: {len(error_cronologia_df)}")
print(f"Registros finales: {len(clean_df)}")

Limpieza: Consistencia cronológica
Registros eliminados: 0
Registros finales: 166749



## NIVEL 3 — Consistencia Cronológica

En este nivel se verifica la coherencia temporal de los proyectos, asegurando un orden lógico entre las fechas clave.

### Lógica de la limpieza

* Se convierten `FECHA_REGISTRO`, `FECHA_VIABILIDAD` y `FEC_INI_EJECUCION` a formato datetime.
* Se valida el orden lógico:

  * `FECHA_REGISTRO` ≤ `FECHA_VIABILIDAD`
  * `FEC_INI_EJECUCION` ≥ `FECHA_VIABILIDAD`
* Los registros que incumplen estas condiciones se consideran inválidos y se eliminan.

### Variables involucradas

* `FECHA_REGISTRO`: Fecha de registro del proyecto.
* `FECHA_VIABILIDAD`: Fecha de aprobación/viabilidad.
* `FEC_INI_EJECUCION`: Fecha de inicio de ejecución.

### Motivos de limpieza:
Los registros con fechas fuera de orden reflejan inconsistencias temporales, como aprobaciones posteriores al inicio de ejecución o registros posteriores a la viabilidad, lo cual contradice la secuencia lógica del ciclo de vida del proyecto.




In [18]:
# nivel 4 - alerta financiera

# transformar a numero
for col in ['SALDO_EJECUTAR', 'COSTO_ACTUALIZADO', 'MONTO_VIABLE']:
    clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')

# trabajamos con errores criticos

mask_financiero_critico = clean_df['SALDO_EJECUTAR'] > clean_df['COSTO_ACTUALIZADO']
error_financiero_df = clean_df[mask_financiero_critico].copy()

# alertas

mask_sobrecosto = clean_df['SALDO_EJECUTAR'] > clean_df['MONTO_VIABLE']
alerta_financiera_df = clean_df[mask_sobrecosto].copy()

# eliminamos alertas y errores

mask_total = mask_financiero_critico | mask_sobrecosto
clean_df = clean_df[~mask_total].copy()

# resultados

print("Limpieza: Consistencia financiera (estricta)")
print(f"Errores críticos eliminados: {len(error_financiero_df)}")
print(f"Alertas eliminadas: {len(alerta_financiera_df)}")
print(f"Total eliminados: {mask_total.sum()}")
print(f"Registros finales: {len(clean_df)}")

Limpieza: Consistencia financiera (estricta)
Errores críticos eliminados: 0
Alertas eliminadas: 26323
Total eliminados: 26323
Registros finales: 173842


## 6 . CASOS DE PRUEBA

In [29]:

#test case del data set original

print("=== VALIDACIÓN DATASET ORIGINAL ===")

# Nivel 2: Estado
test_estado_df = (
    (df['ESTADO'].astype(str).str.strip().str.upper() == 'ACTIVO') &
    (df['SITUACION'].astype(str).str.strip().str.upper() != 'VIABLE')
).sum()

# Nivel 3: Cronologia
df_fechas = df.copy()
for col in ['FECHA_REGISTRO', 'FECHA_VIABILIDAD', 'FEC_INI_EJECUCION']:
    df_fechas[col] = pd.to_datetime(df_fechas[col], errors='coerce')

test_cronologia_df = (
    (df_fechas['FECHA_REGISTRO'] > df_fechas['FECHA_VIABILIDAD']) |
    (df_fechas['FEC_INI_EJECUCION'] < df_fechas['FECHA_VIABILIDAD'])
).sum()

# Nivel 4: Financiero
df_num = df.copy()
for col in ['SALDO_EJECUTAR', 'COSTO_ACTUALIZADO', 'MONTO_VIABLE']:
    df_num[col] = pd.to_numeric(df_num[col], errors='coerce')

test_financiero_df = (
    (df_num['SALDO_EJECUTAR'] > df_num['COSTO_ACTUALIZADO']) |
    (df_num['SALDO_EJECUTAR'] > df_num['MONTO_VIABLE'])
).sum()

print(f"Errores Estado: {test_estado_df}")
print(f"Errores Cronología: {test_cronologia_df}")
print(f"Errores Financieros: {test_financiero_df}")


=== VALIDACIÓN DATASET ORIGINAL ===
Errores Estado: 59843
Errores Cronología: 8195
Errores Financieros: 31822


In [26]:

#test case data set limpio

print("\n=== VALIDACIÓN DATASET LIMPIO ===")

# Nivel 2: Estado
test_estado_clean = (
    (clean_df['ESTADO'] == 'ACTIVO') &
    (clean_df['SITUACION'] != 'VIABLE')
).sum()

# Nivel 3: Cronología
test_cronologia_clean = (
    (clean_df['FECHA_REGISTRO'] > clean_df['FECHA_VIABILIDAD']) |
    (clean_df['FEC_INI_EJECUCION'] < clean_df['FECHA_VIABILIDAD'])
).sum()

# Nivel 4: Financiero
test_financiero_clean = (
    (clean_df['SALDO_EJECUTAR'] > clean_df['COSTO_ACTUALIZADO']) |
    (clean_df['SALDO_EJECUTAR'] > clean_df['MONTO_VIABLE'])
).sum()

print(f"Errores Estado: {test_estado_clean}")
print(f"Errores Cronología: {test_cronologia_clean}")
print(f"Errores Financieros: {test_financiero_clean}")



=== VALIDACIÓN DATASET LIMPIO ===
Errores Estado: 0
Errores Cronología: 0
Errores Financieros: 0


In [28]:
print("=== COMPARACIÓN DE DATASETS ===")
print(f"Registros originales: {len(df)}")
print(f"Registros después de limpieza: {len(clean_df)}")
print(f"Total de registros eliminados: {len(df) - len(clean_df)}")

=== COMPARACIÓN DE DATASETS ===
Registros originales: 260009
Registros después de limpieza: 166749
Total de registros eliminados: 93260
